# Run all experiments

1. Runtime → Change runtime type → **T4 GPU**.
2. Runtime → **Run all**.

Trained fp32 models already on Drive are reused, so this re-run only redoes calibration and quantization (about 1.5 hours). If Colab disconnects, just **Run all** again: finished work is skipped.

Results go to Google Drive: `cnn-study/results_v2.csv`, `cnn-study/calibration_check.csv` and `cnn-study/models/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

In [ ]:
%cd /content
![ -d repo ] && git -C repo pull -q || git clone -q https://github.com/bekadiscoteka/cnn-compression-study repo
%cd /content/repo
!pip install -q -r requirements.txt
!git log --oneline -1

## 1. Choose the calibration method

Compares MinMax and percentile calibration on 5,000 training images that are not used for calibration (the test set is not touched) and writes the choice to `calibration_choice.json`.

In [ ]:
OUT = '/content/drive/MyDrive/cnn-study'
!python calibrate_check.py --out {OUT} 2>&1 | grep -v -E '^(Number of|Percentile|Collecting|Finding)'

## 2. Quantize (PTQ and QAT) and evaluate all 36 models

In [ ]:
!python train.py --out {OUT}

## Results

Mean ± standard deviation of test accuracy over seeds.

In [ ]:
import pandas as pd
df = pd.read_csv(f'{OUT}/results_v2.csv')
print(f'{len(df)}/36 rows')
df.groupby(['pruning', 'quant'])['accuracy'].agg(['mean', 'std', 'count']).round(2).unstack('quant')